# MVDOT Evaluation — `evaluate`

Loads a checkpoint produced by `train_main`, runs consensus clustering over the dataset, and reports clustering accuracy / NMI / ARI. Restructured into stage-wise cells; logic is unchanged from the original script.

## Imports & Path Setup

Bring in the mvdot modules plus sklearn's clustering metrics. *(path setup adjusted for notebook execution — see note below)*

In [1]:
!pip install mvdot.data

ERROR: Could not find a version that satisfies the requirement mvdot.data (from versions: none)
ERROR: No matching distribution found for mvdot.data


In [2]:
import os
import sys

import torch
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score
)

sys.path.append(
    os.path.dirname(
        os.getcwd()
    )
)

from mvdot.data.preprocessing import (
    load_mnist,
    create_noisy_view
)

from mvdot.data.dataset import noisy_mnist
from mvdot.data.dataloader import create_dataloader

from mvdot.models.autoencoder import autoencoder
from mvdot.models.matching import matching_network
from mvdot.models.barycenter import barycenter

from mvdot.inference.consensus import (
    select_reference_view,
    compute_cross_view_plan,
    fuse_batch_representations,
    predict_clusters
)

from mvdot.models.transport_matching import (
    cross_view_affinity,
    build_bipartite_graph,
    bfs_topology,
    topology_cost,
    semantic_cost_from_probabilities,
    cross_view_transport
)

from mvdot.losses.semantic import (
    cluster_probabilities
)


## Helper — Clustering Accuracy (Optimal Label Assignment)

Bitmask DP that finds the best one-to-one mapping between predicted cluster ids and true labels, then returns accuracy under that mapping.

In [3]:
def clustering_accuracy(
    labels,
    predictions,
    num_clusters
):
    confusion = torch.zeros(
        num_clusters,
        num_clusters,
        dtype=torch.long
    )

    for true_label, predicted_label in zip(
        labels,
        predictions
    ):
        confusion[
            predicted_label,
            true_label
        ] += 1

    dp = torch.full(
        (1 << num_clusters,),
        -1,
        dtype=torch.long
    )

    dp[0] = 0

    for mask in range(
        1 << num_clusters
    ):
        used = mask.bit_count()

        if used >= num_clusters:
            continue

        current = dp[
            mask
        ]

        if current < 0:
            continue

        for label in range(
            num_clusters
        ):
            if mask & (
                1 << label
            ):
                continue

            new_mask = (
                mask
                | (
                    1 << label
                )
            )

            value = (
                current
                + confusion[
                    used,
                    label
                ]
            )

            if value > dp[
                new_mask
            ]:
                dp[
                    new_mask
                ] = value

    best = dp[
        (1 << num_clusters) - 1
    ].item()

    return (
        best
        / len(labels)
    )


## Load Checkpoint & Configuration

Load the checkpoint saved by `train_main` and pull out the hyperparameters the model and data pipeline need to be rebuilt identically.

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


checkpoint = torch.load(
    "checkpoints/mvdot_final.pt",
    map_location=device
)


input_dim = checkpoint[
    "input_dim"
]

hidden_dim = checkpoint[
    "hidden_dim"
]

latent_dim = checkpoint[
    "latent_dim"
]

num_clusters = checkpoint[
    "num_clusters"
]

aligned_rate = checkpoint[
    "aligned_rate"
]

sigma = checkpoint[
    "sigma"
]

epsilon = checkpoint[
    "epsilon"
]

top_k = checkpoint[
    "cross_view_top_k"
]

sinkhorn_iterations = checkpoint[
    "sinkhorn_iterations"
]

batch_size = checkpoint[
    "batch_size"
]


## Rebuild Models & Load Trained Weights

Recreate the autoencoders, matching network, and barycenter, then load the trained state dicts into them and switch to eval mode.

In [5]:
model1 = autoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim
).to(device)

model2 = autoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim
).to(device)

matching = matching_network(
    feature_dim=latent_dim
).to(device)

bary = barycenter(
    num_clusters=num_clusters,
    feature_dim=latent_dim
).to(device)


model1.load_state_dict(
    checkpoint["model1"]
)

model2.load_state_dict(
    checkpoint["model2"]
)

matching.load_state_dict(
    checkpoint["matching"]
)

bary.load_state_dict(
    checkpoint["barycenter"]
)


model1.eval()
model2.eval()
matching.eval()
bary.eval()


barycenter()

## Recompute Cluster Centers & Load Evaluation Data

Read out the barycenter's current centers/weights, then rebuild the (unshuffled) noisy-MNIST dataset and loader used for evaluation.

In [6]:
centers, weights = bary()


images, labels = load_mnist(
    "data"
)

noisy_images = create_noisy_view(
    images,
    sigma=sigma,
    seed=checkpoint["seed"]
)


class aligned_noisy_mnist(noisy_mnist):
    def __getitem__(self, index):
        view1, view2, label = super().__getitem__(index)
        return view1, view2, label, self.aligned_mask[index]


dataset = aligned_noisy_mnist(
    images,
    noisy_images,
    labels,
    aligned_rate=aligned_rate,
    seed=checkpoint["seed"]
)


loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False
)


## Select Reference View & Compute Fusion Weights

Run one pass over the data to see which view sits closer to the barycenter, then turn that into a pair of fusion weights for the two views.

In [7]:
reference_cost1 = 0.0
reference_cost2 = 0.0
number_of_batches = 0


print(
    "device:",
    device
)

print(
    "samples:",
    len(dataset)
)

print(
    "aligned rate:",
    aligned_rate
)

print(
    "sigma:",
    sigma
)


print()
print(
    "selecting reference view..."
)


with torch.no_grad():

    for view1, view2, _, _ in loader:

        view1 = view1.to(
            device,
            non_blocking=True
        )

        view2 = view2.to(
            device,
            non_blocking=True
        )

        z1 = model1.encode(
            view1
        )

        z2 = model2.encode(
            view2
        )

        h1 = matching(
            z1
        )

        h2 = matching(
            z2
        )

        t1, cost1 = __import__(
            "mvdot.ot.transport",
            fromlist=["sample_to_cluster_transport"]
        ).sample_to_cluster_transport(
            h1,
            centers,
            weights,
            epsilon=epsilon,
            iterations=sinkhorn_iterations
        )

        t2, cost2 = __import__(
            "mvdot.ot.transport",
            fromlist=["sample_to_cluster_transport"]
        ).sample_to_cluster_transport(
            h2,
            centers,
            weights,
            epsilon=epsilon,
            iterations=sinkhorn_iterations
        )

        reference_cost1 += (
            t1 * cost1
        ).sum().item()

        reference_cost2 += (
            t2 * cost2
        ).sum().item()

        number_of_batches += 1


reference_cost1 /= number_of_batches
reference_cost2 /= number_of_batches


if reference_cost1 <= reference_cost2:
    reference_view = 1
else:
    reference_view = 2


inverse1 = 1.0 / (
    reference_cost1 + 1e-8
)

inverse2 = 1.0 / (
    reference_cost2 + 1e-8
)


view_weight1 = (
    inverse1
    / (
        inverse1
        + inverse2
    )
)

view_weight2 = (
    inverse2
    / (
        inverse1
        + inverse2
    )
)


print(
    "view 1 barycenter cost:",
    reference_cost1
)

print(
    "view 2 barycenter cost:",
    reference_cost2
)

print(
    "reference view:",
    reference_view
)

print(
    "view weights:",
    view_weight1,
    view_weight2
)


device: cpu
samples: 60000
aligned rate: 0.5
sigma: 0.25

selecting reference view...


c:\Users\AYUSH JHA\Miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


view 1 barycenter cost: 7.62932198840451e-05
view 2 barycenter cost: 2.998837912758624e-05
reference view: 2
view weights: 0.2822006386216624 0.7177993613783376


## Consensus Clustering — Generate Predictions

For every batch, align the two views via cross-view transport, fuse them using the view weights, and assign each sample to its nearest cluster center.

In [8]:
predictions = []
true_labels = []


print()
print(
    "performing consensus clustering..."
)


with torch.no_grad():

    for batch_index, batch in enumerate(loader):
        view1, view2, batch_labels, is_aligned = batch

        view1 = view1.to(
            device,
            non_blocking=True
        )

        view2 = view2.to(
            device,
            non_blocking=True
        )

        is_aligned = is_aligned.to(device)

        z1 = model1.encode(
            view1
        )

        z2 = model2.encode(
            view2
        )

        h1 = matching(
            z1
        )

        h2 = matching(
            z2
        )

        p12 = compute_cross_view_plan(
            h1,
            h2,
            centers,
            epsilon,
            sinkhorn_iterations,
            top_k
        )

        fused = fuse_batch_representations(
            h1,
            h2,
            is_aligned,
            reference_view,
            view_weight1,
            view_weight2,
            p12
        )

        batch_predictions = predict_clusters(
            fused,
            centers
        )

        predictions.append(
            batch_predictions.cpu()
        )

        true_labels.append(
            batch_labels
        )

        if (
            batch_index == 0
            or (batch_index + 1) % 25 == 0
            or batch_index + 1 == len(loader)
        ):
            print(
                "batch:",
                batch_index + 1,
                "/",
                len(loader)
            )



performing consensus clustering...
batch: 1 / 235
batch: 25 / 235
batch: 50 / 235
batch: 75 / 235
batch: 100 / 235
batch: 125 / 235
batch: 150 / 235
batch: 175 / 235
batch: 200 / 235
batch: 225 / 235
batch: 235 / 235


## Compute & Report Final Metrics

Score the predictions against the ground-truth labels with clustering accuracy, NMI, and ARI.

In [9]:
acc = clustering_accuracy(
    torch.tensor(true_labels),
    torch.tensor(predictions),
    num_clusters
)

nmi = normalized_mutual_info_score(
    true_labels,
    predictions
)

ari = adjusted_rand_score(
    true_labels,
    predictions
)


print()
print(
    "final results"
)

print(
    "samples:",
    len(true_labels)
)

print(
    "reference view:",
    reference_view
)

print(
    "view weight 1:",
    view_weight1
)

print(
    "view weight 2:",
    view_weight2
)

print()
print(
    "acc:",
    acc
)

print(
    "nmi:",
    nmi
)

print(
    "ari:",
    ari
)




final results
samples: 60000
reference view: 2
view weight 1: 0.2822006386216624
view weight 2: 0.7177993613783376

acc: 0.10566666666666667
nmi: 0.0003190767080872651
ari: 2.799381708269933e-05
